# 导入相关库和文件

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud

In [ ]:
# 使用GBK编码读取数据集
df = pd.read_csv('题目4数据.csv', encoding='GBK')
df = df.drop_duplicates()
# 将 salaryMonth 为 0 的值替换为 12
df['salaryMonth'] = df['salaryMonth'].replace(0, 12)

# 查看文件基本信息

In [ ]:
print('数据基本信息：')
df.info()

In [ ]:
# 查看数据集行数和列数
rows, columns = df.shape

In [ ]:
print('数据前几行内容信息：')
print(df.head())

# 薪资相关

## 1.薪资分析

### 1）绘制箱线图展示年薪的下限和上限分布情况

In [ ]:
# 设置图片清晰度
plt.rcParams['figure.dpi'] = 800
# 支持中文
plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['font.sans-serif'] = ['SimHei'] 
# 负数乱码
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 提取薪资下限和上限
df[['salary_lower', 'salary_upper']] = df['salary'].str.extract(r'(\d+)k-(\d+)k').astype(float)

# 计算年薪下限和上限
df['annual_salary_lower'] = df['salary_lower'] * df['salaryMonth']
df['annual_salary_upper'] = df['salary_upper'] * df['salaryMonth']

# 创建画布
plt.figure(figsize=(8, 6))

# 绘制年薪下限箱线图
plt.subplot(1, 2, 1)
df['annual_salary_lower'].plot.box()

# 设置标题和坐标轴标签
plt.title('年薪下限箱线图')
plt.ylabel('年薪（千元）')

# 绘制年薪上限箱线图
plt.subplot(1, 2, 2)
df['annual_salary_upper'].plot.box()

# 设置标题
plt.title('年薪上限箱线图')

# 显示图形
plt.show()

### 2）薪资分析可视化

In [ ]:
# 处理薪资列，提取薪资下限和上限
df['salary'] = df['salary'].str.replace('k', '000')
df['salary_lower'] = df['salary'].str.extract(r'(\d+)').astype(int)
df['salary_upper'] = df['salary'].str.extract(r'-(\d+)')

# 处理缺失值，将 NaN 替换为 5000
df['salary_upper'] = df['salary_upper'].fillna(5000).astype(int)

# 计算平均薪资
df['salary_avg'] = (df['salary_lower'] + df['salary_upper']) / 2

# 定义薪资区间（以千为单位）
bins = [0, 5, 10, 15, 20, 25, float('inf')]
labels = ['0k ～ 5k', '5k ～ 10k', '10k ～ 15k', '15k ～ 20k', '20k ～ 25k', '25k以上']
df['salary_range'] = pd.cut(df['salary_avg'] / 1000, bins=bins, labels=labels)

# 统计各薪资区间的数量
salary_range_counts = df['salary_range'].value_counts().sort_index()

# 创建画布，包含两个子图
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 绘制柱状图
bars = axes[0].bar(salary_range_counts.index, salary_range_counts.values, color=plt.cm.Paired.colors)
axes[0].set_title('各薪资区间的数量分布(柱状图)', color = 'r')
axes[0].set_xlabel('薪资区间', color = 'purple')
axes[0].set_ylabel('数量(个)', color = 'purple')
axes[0].tick_params(axis='x', rotation = 45)

# 在柱状图上添加数据标签
for bar in bars:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width() / 2, height, f'{height}', ha='center', va='bottom')

# 绘制饼图
axes[1].pie(salary_range_counts.values, labels=salary_range_counts.index, autopct='%1.1f%%')
axes[1].set_title('各薪资区间的数量分布(饼图)', color = 'r')

plt.tight_layout()
plt.show()

### 3）薪资总体分析

In [ ]:
# 使用GBK编码读取数据集
df = pd.read_csv('题目4数据.csv', encoding='GBK')
df = df.drop_duplicates()
# 将 salaryMonth 为 0 的值替换为 12
df['salaryMonth'] = df['salaryMonth'].replace(0, 12)

# 设置图片清晰度
plt.rcParams['figure.dpi'] = 800
# 支持中文
plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['font.sans-serif'] = ['SimHei'] 
# 负数乱码
plt.rcParams['axes.unicode_minus'] = False

# 提取薪资下限（支持"Xk-Yk"格式）
df['salary_lower'] = df['salary'].str.extract(r'(\d+)k').astype(float)  # 匹配任意位置的"数字k"
# 提取薪资上限（仅当存在"-"时提取）
df['salary_upper'] = df['salary'].str.extract(r'-(\d+)k').astype(float)  # 匹配"-数字k"

# 计算年薪
df['annual_salary_lower'] = df['salary_lower'] * df['salaryMonth']
df['annual_salary_upper'] = df['salary_upper'] * df['salaryMonth']

# 打印年薪的描述性统计信息
print("年薪下限描述性统计信息：")
print(df['annual_salary_lower'].describe().round(2))
print("\n年薪上限描述性统计信息：")
print(df['annual_salary_upper'].describe().round(2))

# 二、不同城市的平均年薪
# 按城市分组，计算年薪下限和上限的平均值
city_salary = df.groupby('city')[['annual_salary_lower', 'annual_salary_upper']].mean()
print("\n不同城市的平均年薪：")
print(city_salary.round(2))

# 三、不同工作年限的平均年薪
# 按工作年限分组，计算年薪下限和上限的平均值
work_year_salary = df.groupby('workYear', observed=False)[['annual_salary_lower', 'annual_salary_upper']].mean()
print("\n不同工作年限的平均年薪：")
print(work_year_salary.round(2))

# 四、N 薪分布
# 统计不同 salaryMonth 的职位数量
salary_month_distribution = df['salaryMonth'].value_counts().sort_index()

# 将结果转换为 DataFrame 并设置列名
result_df = pd.DataFrame(salary_month_distribution)
result_df.columns = ['数量']

print("\nN 薪分布：")
print(result_df)

## 2.不同城市薪资分布

In [ ]:
# 使用GBK编码读取数据集
df = pd.read_csv('题目4数据.csv', encoding='GBK')
df = df.drop_duplicates()
# 将 salaryMonth 为 0 的值替换为 12
df['salaryMonth'] = df['salaryMonth'].replace(0, 12)

# 设置图片清晰度
plt.rcParams['figure.dpi'] = 800
# 支持中文
plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['font.sans-serif'] = ['SimHei'] 
# 负数乱码
plt.rcParams['axes.unicode_minus'] = False

# 提取薪资下限（支持"Xk-Yk"格式）
df['salary_lower'] = df['salary'].str.extract(r'(\d+)k').astype(float)  # 匹配任意位置的"数字k"
# 提取薪资上限（仅当存在"-"时提取）
df['salary_upper'] = df['salary'].str.extract(r'-(\d+)k').astype(float)  # 匹配"-数字k"

# 计算年薪
df['annual_salary_lower'] = df['salary_lower'] * df['salaryMonth']
df['annual_salary_upper'] = df['salary_upper'] * df['salaryMonth']

# 按城市分组，计算年薪下限和上限的平均值
city_salary = df.groupby('city')[['annual_salary_lower', 'annual_salary_upper']].mean()

# 创建画布
plt.figure(figsize=(12, 8))

# 绘制年薪下限柱状图
bars_lower = plt.bar(city_salary.index, city_salary['annual_salary_lower'], width=-0.4, align='edge', label='年薪下限', color='blue')

# 绘制年薪上限柱状图
bars_upper = plt.bar(city_salary.index, city_salary['annual_salary_upper'], width=0.4, align='edge', label='年薪上限', color='orange')

# 添加数据标签
for bar in bars_lower:
    height = bar.get_height()
    plt.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3), textcoords='offset points', ha='center', va='bottom')
for bar in bars_upper:
    height = bar.get_height()
    plt.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3), textcoords='offset points', ha='center', va='bottom')

# 设置标题和坐标轴标签
plt.title('不同城市的平均年薪')
plt.xlabel('城市')
plt.ylabel('平均年薪(千元)')

# 设置 x 轴刻度旋转角度
plt.xticks(rotation=45)

# 显示图例
plt.legend()

# 显示图形
plt.show()

## 3.N薪分布

In [ ]:
# 使用GBK编码读取数据集
df = pd.read_csv('题目4数据.csv', encoding='GBK')
df = df.drop_duplicates()
# 将 salaryMonth 为 0 的值替换为 12
df['salaryMonth'] = df['salaryMonth'].replace(0, 12)

# 设置图片清晰度
plt.rcParams['figure.dpi'] = 800
# 支持中文
plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['font.sans-serif'] = ['SimHei'] 
# 负数乱码
plt.rcParams['axes.unicode_minus'] = False

# 提取薪资下限（支持"Xk-Yk"格式）
df['salary_lower'] = df['salary'].str.extract(r'(\d+)k').astype(float)  # 匹配任意位置的"数字k"
# 提取薪资上限（仅当存在"-"时提取）
df['salary_upper'] = df['salary'].str.extract(r'-(\d+)k').astype(float)  # 匹配"-数字k"

# 计算年薪
df['annual_salary_lower'] = df['salary_lower'] * df['salaryMonth']
df['annual_salary_upper'] = df['salary_upper'] * df['salaryMonth']

# 统计不同 salaryMonth 的职位数量
salary_month_distribution = df['salaryMonth'].value_counts().sort_index()

# 创建画布
plt.figure(figsize=(10, 6))

# 绘制柱状图
bars = plt.bar(salary_month_distribution.index, salary_month_distribution.values)

# 添加数据标签
for bar in bars:
    height = bar.get_height()
    plt.annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3), textcoords='offset points', ha='center', va='bottom')

# 设置标题和坐标轴标签
plt.title('N 薪分布')
plt.xlabel('N 薪')
plt.xticks(rotation=45)
plt.ylabel('职位数量')

# 显示图形
plt.show()

## 4.工作年限与薪资

In [ ]:
# 使用GBK编码读取数据集
df = pd.read_csv('题目4数据.csv', encoding='GBK')
df = df.drop_duplicates()
# 将 salaryMonth 为 0 的值替换为 12
df['salaryMonth'] = df['salaryMonth'].replace(0, 12)

# 设置图片清晰度
plt.rcParams['figure.dpi'] = 800
# 支持中文
plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['font.sans-serif'] = ['SimHei'] 
# 负数乱码
plt.rcParams['axes.unicode_minus'] = False

# 提取薪资下限（支持"Xk-Yk"格式）
df['salary_lower'] = df['salary'].str.extract(r'(\d+)k').astype(float)  # 匹配任意位置的"数字k"
# 提取薪资上限（仅当存在"-"时提取）
df['salary_upper'] = df['salary'].str.extract(r'-(\d+)k').astype(float)  # 匹配"-数字k"

# 计算年薪
df['annual_salary_lower'] = df['salary_lower'] * df['salaryMonth']
df['annual_salary_upper'] = df['salary_upper'] * df['salaryMonth']

# 定义工作年限顺序
work_year_order = ['不限', '在校/应届', '1年以下', '1-3年', '3-5年', '5-10年', '10年以上']

# 清洗薪资字段
# 提取薪资下限（支持"Xk-Yk"格式）
df['salary_lower'] = df['salary'].str.extract(r'(\d+)k').astype(float)  # 匹配任意位置的"数字k"
# 提取薪资上限（仅当存在"-"时提取）
df['salary_upper'] = df['salary'].str.extract(r'-(\d+)k').astype(float)  # 匹配"-数字k"

# 计算年薪
df['annual_salary_lower'] = df['salary_lower'] * df['salaryMonth']
df['annual_salary_upper'] = df['salary_upper'] * df['salaryMonth']

# 转换工作年限为有序分类变量
df['workYear'] = pd.Categorical(df['workYear'], categories=work_year_order, ordered=True)

# 分组计算
work_year_salary = df.groupby('workYear', observed=False)[['annual_salary_lower', 'annual_salary_upper']].mean()
work_year_salary = work_year_salary.reindex(work_year_order)  # 确保顺序

# 绘图部分
plt.figure(figsize=(10, 6))
plt.rcParams.update({
    'figure.dpi': 800,
    'font.sans-serif': ['SimHei'],
    'axes.unicode_minus': False
})

plt.plot(work_year_salary.index, work_year_salary['annual_salary_lower'], marker='o', linewidth=2, label='平均年薪下限')
plt.plot(work_year_salary.index, work_year_salary['annual_salary_upper'], marker='s', linewidth=2, label='平均年薪上限')

# 数据标签
for x, y in zip(work_year_salary.index, work_year_salary['annual_salary_lower']):
    if pd.notna(y):
        plt.annotate(f'{y:.1f}', (x, y), xytext=(0, 5), textcoords='offset points', ha='center', fontsize=9)
    
for x, y in zip(work_year_salary.index, work_year_salary['annual_salary_upper']):
    if pd.notna(y):
        plt.annotate(f'{y:.1f}', (x, y), xytext=(0, 5), textcoords='offset points', ha='center', fontsize=9)

plt.title('不同工作年限的平均年薪分布', fontsize=14, pad=20)
plt.xlabel('工作年限', fontsize=12)
plt.ylabel('平均年薪（千元）', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(fontsize=10, frameon=True, shadow=True)
plt.tight_layout()
plt.show()

# 职位分析

## 1.不同城市职位数量

In [ ]:
# 使用GBK编码读取数据集
df = pd.read_csv('题目4数据.csv', encoding='GBK')
df = df.drop_duplicates()
# 将 salaryMonth 为 0 的值替换为 12
df['salaryMonth'] = df['salaryMonth'].replace(0, 12)

# 设置图片清晰度
plt.rcParams['figure.dpi'] = 800
# 支持中文
plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['font.sans-serif'] = ['SimHei'] 
# 负数乱码
plt.rcParams['axes.unicode_minus'] = False

# 提取薪资下限（支持"Xk-Yk"格式）
df['salary_lower'] = df['salary'].str.extract(r'(\d+)k').astype(float)  # 匹配任意位置的"数字k"
# 提取薪资上限（仅当存在"-"时提取）
df['salary_upper'] = df['salary'].str.extract(r'-(\d+)k').astype(float)  # 匹配"-数字k"

# 计算年薪
df['annual_salary_lower'] = df['salary_lower'] * df['salaryMonth']
df['annual_salary_upper'] = df['salary_upper'] * df['salaryMonth']

# 统计不同城市的职位数量
city_counts = df['city'].value_counts().reset_index(name='职位数量')

# 为每个柱子设置不同颜色
colors = ['purple', 'r', 'y', 'b', 'c', 'm', 'orange', 'g', 'brown']

# 绘制柱状图
plt.bar(city_counts['city'], city_counts['职位数量'], color=colors)

# 设置标题和坐标轴标签
plt.title('不同城市职位数量(柱状图)')
plt.xlabel('城市')
plt.ylabel('职位数量')

# 添加数据标签，调整垂直偏移量，这里将偏移量设为 10
for i, v in enumerate(city_counts['职位数量']):
    plt.text(i, v + 5, str(v), ha='center')

# 显示图形
plt.show()

## 2.公司融资情况

In [ ]:
# 使用GBK编码读取数据集
df = pd.read_csv('题目4数据.csv', encoding='GBK')
df = df.drop_duplicates()
# 将 salaryMonth 为 0 的值替换为 12
df['salaryMonth'] = df['salaryMonth'].replace(0, 12)

# 设置图片清晰度
plt.rcParams['figure.dpi'] = 800
# 支持中文
plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['font.sans-serif'] = ['SimHei'] 
# 负数乱码
plt.rcParams['axes.unicode_minus'] = False

# 提取薪资下限（支持"Xk-Yk"格式）
df['salary_lower'] = df['salary'].str.extract(r'(\d+)k').astype(float)  # 匹配任意位置的"数字k"
# 提取薪资上限（仅当存在"-"时提取）
df['salary_upper'] = df['salary'].str.extract(r'-(\d+)k').astype(float)  # 匹配"-数字k"

# 计算年薪
df['annual_salary_lower'] = df['salary_lower'] * df['salaryMonth']
df['annual_salary_upper'] = df['salary_upper'] * df['salaryMonth']

# 创建画布，设置子图布局
fig, axes = plt.subplots(3, 1, figsize=(10, 15))

# 调整子图之间的垂直间距
# plt.subplots_adjust(hspace=0.5)

# 一、不同融资阶段的公司数量柱状图
# 统计不同融资情况的公司数量
finance_counts = df['financeStage'].value_counts().reset_index(name='公司数量')

# 为每个柱子设置不同颜色
colors = ['r', 'y', 'b', 'c', 'm', 'orange', 'purple', 'brown']

# 绘制柱状图
axes[0].bar(finance_counts['financeStage'], finance_counts['公司数量'], color = colors)

# 设置标题和坐标轴标签
axes[0].set_title('不同融资情况的公司数量分布', color = 'r')
axes[0].set_xlabel('融资情况')
axes[0].set_ylabel('公司数量')

# 添加数据标签
for i, v in enumerate(finance_counts['公司数量']):
    axes[0].text(i, v+5, str(v), ha='center')

# 旋转 x 轴标签以便更好显示
axes[0].tick_params(axis='x', rotation=45)

# 二、不同融资阶段的平均薪资关系图
# 计算不同融资阶段的平均年薪下限和上限
finance_salary = df.groupby('financeStage')[['annual_salary_lower', 'annual_salary_upper']].mean().reset_index()

# 绘制平均年薪下限折线图
axes[1].plot(finance_salary['financeStage'], finance_salary['annual_salary_lower'], marker='o', label='平均年薪下限')

# 绘制平均年薪上限折线图
axes[1].plot(finance_salary['financeStage'], finance_salary['annual_salary_upper'], marker='s', label='平均年薪上限')

# 添加数据标签
for x, y in zip(finance_salary['financeStage'], finance_salary['annual_salary_lower']):
    axes[1].annotate(f'{y:.1f}', (x, y), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=9)
for x, y in zip(finance_salary['financeStage'], finance_salary['annual_salary_upper']):
    axes[1].annotate(f'{y:.1f}', (x, y), xytext=(0, -15), textcoords='offset points', ha='center', fontsize=9)

# 设置标题和坐标轴标签
axes[1].set_title('不同融资阶段公司员工的平均年薪分布', color = 'r')
axes[1].set_xlabel('融资阶段')
axes[1].set_ylabel('平均年薪（千元）')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

# 三、不同融资阶段公司占比饼图
# 计算不同融资阶段公司的占比
total_companies = finance_counts['公司数量'].sum()
finance_counts['占比(%)'] = (finance_counts['公司数量'] / total_companies * 100).round(2)

# 绘制饼图
axes[2].pie(finance_counts['公司数量'], labels=finance_counts['financeStage'], autopct='%1.2f%%', startangle=140)
axes[2].set_title('不同融资阶段公司占比', color = 'r')

plt.tight_layout()
plt.show()

## 3.不同行业公司数量

In [ ]:
# 使用GBK编码读取数据集
df = pd.read_csv('题目4数据.csv', encoding='GBK')
df = df.drop_duplicates()
# 将 salaryMonth 为 0 的值替换为 12
df['salaryMonth'] = df['salaryMonth'].replace(0, 12)

# 设置图片清晰度
plt.rcParams['figure.dpi'] = 800
# 支持中文
plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['font.sans-serif'] = ['SimHei'] 
# 负数乱码
plt.rcParams['axes.unicode_minus'] = False

# 提取薪资下限（支持"Xk-Yk"格式）
df['salary_lower'] = df['salary'].str.extract(r'(\d+)k').astype(float)  # 匹配任意位置的"数字k"
# 提取薪资上限（仅当存在"-"时提取）
df['salary_upper'] = df['salary'].str.extract(r'-(\d+)k').astype(float)  # 匹配"-数字k"

# 计算年薪
df['annual_salary_lower'] = df['salary_lower'] * df['salaryMonth']
df['annual_salary_upper'] = df['salary_upper'] * df['salaryMonth']

# 统计行业数量
industry_num = df['industryField'].nunique()
print('行业数量：', industry_num)

&emsp;&emsp;总共有389个行业，全部使用柱状图分析不合适，这里仅筛选出公司数量前 10 的行业，再绘制柱状图进行可视化分析。

In [ ]:
# 统计不同行业的公司数量
industry_counts = df['industryField'].value_counts().reset_index(name='公司数量')

# 筛选出公司数量前10的行业
top10_industry = industry_counts.nlargest(10, '公司数量')

# 创建画布
plt.figure(figsize=(10, 6))

# 绘制公司数量前 10 的行业柱状图，设置不同颜色
bar = plt.bar(top10_industry['industryField'], top10_industry['公司数量'], color=plt.cm.Paired.colors)

# 显示数值
plt.bar_label(bar)

# 设置标题和坐标轴标签
plt.title('公司数量位于前10的行业(柱状图)', color='r')
plt.xlabel('行业', color='purple')
plt.ylabel('公司数量', color='purple')
plt.xticks(rotation=45)

# 显示图形
plt.show()

## 4.工作经验分析

In [ ]:
# 使用GBK编码读取数据集
df = pd.read_csv('题目4数据.csv', encoding='GBK')
df = df.drop_duplicates()
# 将 salaryMonth 为 0 的值替换为 12
df['salaryMonth'] = df['salaryMonth'].replace(0, 12)

# 设置图片清晰度
plt.rcParams['figure.dpi'] = 800
# 支持中文
plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['font.sans-serif'] = ['SimHei'] 
# 负数乱码
plt.rcParams['axes.unicode_minus'] = False

# 提取薪资下限（支持"Xk-Yk"格式）
df['salary_lower'] = df['salary'].str.extract(r'(\d+)k').astype(float)  # 匹配任意位置的"数字k"
# 提取薪资上限（仅当存在"-"时提取）
df['salary_upper'] = df['salary'].str.extract(r'-(\d+)k').astype(float)  # 匹配"-数字k"

# 计算年薪
df['annual_salary_lower'] = df['salary_lower'] * df['salaryMonth']
df['annual_salary_upper'] = df['salary_upper'] * df['salaryMonth']

# 统计不同工作经验的数量
workYear_counts = df['workYear'].value_counts().reset_index(name='数量')

# 定义工作经验的顺序
workYear_order = ['不限', '在校/应届', '1年以下', '1-3年', '3-5年', '5-10年', '10年以上']

# 按照指定顺序对数据进行排序
workYear_counts['workYear'] = pd.Categorical(workYear_counts['workYear'], categories=workYear_order, ordered=True)
workYear_counts = workYear_counts.sort_values('workYear')

# 创建画布
plt.figure(figsize=(10, 6))

# 绘制不同工作经验的数量柱状图，设置不同颜色
bar = plt.bar(workYear_counts['workYear'], workYear_counts['数量'], color=plt.cm.Paired.colors)

# 显示数值
plt.bar_label(bar)

# 设置标题和坐标轴标签
plt.title('不同工作经验的数量(柱状图)', color = 'r')
plt.xlabel('工作经验', color = 'purple')
plt.ylabel('数量', color = 'purple')
plt.xticks(rotation = 45)

# 显示图形
plt.show()

In [ ]:
# 使用GBK编码读取数据集
df = pd.read_csv('题目4数据.csv', encoding='GBK')
df = df.drop_duplicates()
# 将 salaryMonth 为 0 的值替换为 12
df['salaryMonth'] = df['salaryMonth'].replace(0, 12)

# 设置图片清晰度
plt.rcParams['figure.dpi'] = 800
# 支持中文
plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['font.sans-serif'] = ['SimHei'] 
# 负数乱码
plt.rcParams['axes.unicode_minus'] = False

# 提取薪资下限（支持"Xk-Yk"格式）
df['salary_lower'] = df['salary'].str.extract(r'(\d+)k').astype(float)  # 匹配任意位置的"数字k"
# 提取薪资上限（仅当存在"-"时提取）
df['salary_upper'] = df['salary'].str.extract(r'-(\d+)k').astype(float)  # 匹配"-数字k"

# 计算年薪
df['annual_salary_lower'] = df['salary_lower'] * df['salaryMonth']
df['annual_salary_upper'] = df['salary_upper'] * df['salaryMonth']

# 创建画布
plt.figure(figsize=(10, 6))

# 绘制不同工作经验的数量折线图，设置不同颜色
plt.plot(workYear_counts['workYear'], workYear_counts['数量'], marker='o', color=plt.cm.Paired.colors[0])

# 在折点上显示数值
for x, y in zip(workYear_counts['workYear'], workYear_counts['数量']):
    plt.annotate(f'{y}', (x, y), textcoords='offset points', xytext=(0,-10), ha='center')

# 设置标题和坐标轴标签
plt.title('不同工作经验的数量(折线图)', color = 'r')
plt.xlabel('工作经验', color = 'purple')
plt.ylabel('数量', color = 'purple')
plt.xticks(rotation = 45)

# 显示图形
plt.show()

## 5.职位标签:词云制作

In [ ]:
# 使用GBK编码读取数据集
df = pd.read_csv('题目4数据.csv', encoding='GBK')
df = df.drop_duplicates()
# 将 salaryMonth 为 0 的值替换为 12
df['salaryMonth'] = df['salaryMonth'].replace(0, 12)

# 设置图片清晰度
plt.rcParams['figure.dpi'] = 800
# 支持中文
plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['font.sans-serif'] = ['SimHei'] 
# 负数乱码
plt.rcParams['axes.unicode_minus'] = False

# 提取薪资下限（支持"Xk-Yk"格式）
df['salary_lower'] = df['salary'].str.extract(r'(\d+)k').astype(float)  # 匹配任意位置的"数字k"
# 提取薪资上限（仅当存在"-"时提取）
df['salary_upper'] = df['salary'].str.extract(r'-(\d+)k').astype(float)  # 匹配"-数字k"

# 计算年薪
df['annual_salary_lower'] = df['salary_lower'] * df['salaryMonth']
df['annual_salary_upper'] = df['salary_upper'] * df['salaryMonth']

# 提取职位标签列并合并文本
text = ' '.join(df['positionLables'])

# 配置参数
plt.rcParams['figure.dpi'] = 1200  # 提高图片精度
font_path = 'SimHei.ttf'

# 创建词云（关键改进点）
wordcloud = WordCloud(
    background_color = 'white',
    width = 800,  # 增大画布尺寸
    height = 400,
    font_path = font_path,
    colormap = 'rainbow',  # 使用彩虹色映射
    min_font_size = 8,    # 最小字体大小
    max_font_size = 120,  # 最大字体大小
    random_state = 24     # 固定随机种子保证颜色一致性
).generate(text)

# 显示词云
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')

# 保存图片（新增功能）
# save_path = '/职位标签词云.png'  # 保存路径
# wordcloud.to_file(save_path)  # 直接通过WordCloud对象保存高清单图
# print(f'词云已保存至：{save_path}')

# plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
plt.show()